<a href="https://colab.research.google.com/github/arturan2101/MedlinePlus-RAG/blob/main/medlineplus_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MedlinePlus RAG Health Assistant

### Project Overview
This project builds a fully local Retrieval-Augmented Generation (RAG) system that answers health-related questions using real data from MedlinePlus — the U.S. National Library of Medicine's consumer health encyclopedia.
The system requires no paid API calls for inference. All components run inside Google Colab, making it accessible and reproducible.

### Block 1 — Installing Dependencies
**What this block does**
This is the setup cell that installs all required Python libraries into the Colab environment. It only needs to be run once per session.


In [ ]:
# Install required libraries (run once)
!pip install -q --upgrade langchain langchain-community gradio langchain-text-splitters sentence-transformers faiss-gpu-cu12

### Block 2 — Importing Libraries
**What this block does**
This block imports all the modules needed throughout the project. Each import corresponds to a specific role in the pipeline.

**Why each import matters:**
* **gradio** — Creates the chat interface that users interact with
* **FAISS** — Manages the vector database for similarity search
* **HuggingFaceEmbeddings** — Converts text into numerical vectors
* **transformers.pipeline** — Loads and runs the Qwen language model
* **requests + lxml + BeautifulSoup** — Used to fetch and parse the MedlinePlus XML data
* **RecursiveCharacterTextSplitter** — Intelligently splits long texts into manageable chunks

In [ ]:
#Import libraries
import gradio as gr

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import pipeline


### Block 3 — Initializing the Retrieval System
**What this block does**
This block initializes the local vector database (FAISS), which acts as the "memory" for our RAG system. It prepares the environment to perform semantic searches against the MedlinePlus data.

**Technical Details:**
* **Embedding Model Initialization**: Loads the `HuggingFaceEmbeddings` model. This model is responsible for converting natural language queries into high-dimensional numerical vectors.
* **Index Loading**: The `FAISS.load_local` function retrieves the pre-built search index from the `faiss_Health` directory. This index contains the processed medical knowledge chunks.
* **Deserialization**: The `allow_dangerous_deserialization=True` flag is enabled to permit the loading of local pickle files, which is the standard storage format for FAISS indexes.

By the end of this block, the system is ready to receive user questions and retrieve the most relevant medical contexts.

In [ ]:
# Load the same embedding model used when creating the FAISS index
embedding_model = HuggingFaceEmbeddings()

# Load the saved FAISS index
db = FAISS.load_local(
    "faiss_Health",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS index loaded successfully.")

In [ ]:
query = "symptoms of diabetes"

results = db.similarity_search(query, k=5)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(r.page_content[:500])

In [ ]:
# Combine the retrieved chunks into one context
context = "\n\n".join([r.page_content for r in results])

print(context[:1000])

### Block 4 — Initializing the Generative Model
**What this block does**
This block loads the Large Language Model (LLM) that will be used to generate human-like responses based on the retrieved medical context.

**Technical Details:**
* **Model Selection**: We use `Qwen/Qwen2.5-0.5B-Instruct`, a lightweight and efficient model optimized for following instructions while fitting within Colab's memory constraints.
* **Pipeline Creation**: The `pipeline("text-generation")` function from the Transformers library abstracts the complexity of loading weights and configuration, setting up a ready-to-use interface for inference.
* **Tokenizer Initialization**: The tokenizer is loaded alongside the model to ensure that user text is converted into the specific numerical format (tokens) that the Qwen model understands.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

generator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name
)

print("Model loaded successfully.")

In [ ]:
# Save FAISS index locally
db.save_local("faiss_Health")

### Block 5 — Launching the Health Assistant Interface
**What this block does**
This final block integrates the retrieval and generation components into a unified chat function and launches a web-based user interface using Gradio.

**Technical Details:**
* **RAG Chat Logic**: The `rag_chat` function performs a similarity search to find the top 5 relevant context chunks, constructs a strictly grounded prompt, and utilizes the LLM to generate an answer.
* **Prompt Engineering**: We use a system-level instruction to ensure the model only answers based on the provided context and always includes a medical disclaimer.
* **Inference Parameters**: The model is set to `temperature=0.1` to maximize factual accuracy and minimize hallucinations.
* **Gradio UI**: A specialized `ChatInterface` is launched, providing a clean, accessible frontend for interacting with the local RAG system. It also generates a shareable public URL for remote testing.

In [ ]:
def rag_chat(message, history):
    # 1. Retrieval: find relevant chanks from FAISS
    results = db.similarity_search(message, k=5)
    context = "\n\n".join([r.page_content for r in results])

    # 2. Formulate promt for Qwen Instruct
    prompt = f"""<|im_start|>system
You are a helpful medical information assistant. Answer ONLY based on the provided context.
If the answer is not in the context, say: "I don't have information about that."
Always recommend consulting a doctor for personal medical advice.
<|im_end|>
<|im_start|>user
Context:
{context}

Question: {message}
<|im_end|>
<|im_start|>assistant
"""

    # 3. Generate the answers from Qwen
    output = generator(
        prompt,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.1,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    # 4. Omit only new text (no promt)
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):].strip()

    # Split if model starts repeting
    if "<|im_end|>" in answer:
        answer = answer.split("<|im_end|>")[0].strip()

    return answer

# Gradio interface
with gr.Blocks(title="MedlinePlus Health Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# MedlinePlus Health Assistant")
    gr.Markdown("Ask questions based on MedlinePlus health data. *Always consult a doctor for personal advice.*")

    gr.ChatInterface(
        fn=rag_chat,
        examples=[
            "What are the symptoms of diabetes?",
            "How to treat high blood pressure?",
            "What is A1C test?",
            "What causes heart disease?",
        ],
        cache_examples=False,
    )

demo.launch(share=True)